#  Learning Unsupervised Embeddings for Molecules

In this tutorial, we will use a `SeqToSeq` model to generate fingerprints for classifying molecules.  This is based on the following paper, although some of the implementation details are different: Xu et al., "Seq2seq Fingerprint: An Unsupervised Deep Molecular Embedding for Drug Discovery" (https://doi.org/10.1145/3107411.3107424).

## Colab

This tutorial and the rest in this sequence can be done in Google colab. If you'd like to open this notebook in colab, you can use the following link.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/deepchem/deepchem/blob/master/examples/tutorials/Learning_Unsupervised_Embeddings_for_Molecules.ipynb)



In [14]:
!pip install --pre deepchem
import deepchem
deepchem.__version__

DEPRECATION: Loading egg at /home/jantine/miniconda3/envs/deepchem/lib/python3.12/site-packages/sympy-1.13.3-py3.12.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /home/jantine/miniconda3/envs/deepchem/lib/python3.12/site-packages/pillow-11.1.0-py3.12-linux-x86_64.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /home/jantine/miniconda3/envs/deepchem/lib/python3.12/site-packages/rdkit-2024.9.5-py3.12-linux-x86_64.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /home/jant

'2.5.0'

In [15]:
import deepchem as dc
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ExponentialLR

In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


# Learning Embeddings with SeqToSeq

Many types of models require their inputs to have a fixed shape.  Since molecules can vary widely in the numbers of atoms and bonds they contain, this makes it hard to apply those models to them.  We need a way of generating a fixed length "fingerprint" for each molecule.  Various ways of doing this have been designed, such as the Extended-Connectivity Fingerprints (ECFPs) we used in earlier tutorials.  But in this example, instead of designing a fingerprint by hand, we will let a `SeqToSeq` model learn its own method of creating fingerprints.

A `SeqToSeq` model performs sequence to sequence translation.  For example, they are often used to translate text from one language to another.  It consists of two parts called the "encoder" and "decoder".  The encoder is a stack of recurrent layers.  The input sequence is fed into it, one token at a time, and it generates a fixed length vector called the "embedding vector".  The decoder is another stack of recurrent layers that performs the inverse operation: it takes the embedding vector as input, and generates the output sequence.  By training it on appropriately chosen input/output pairs, you can create a model that performs many sorts of transformations.

In this case, we will use SMILES strings describing molecules as the input sequences.  We will train the model as an autoencoder, so it tries to make the output sequences identical to the input sequences.  For that to work, the encoder must create embedding vectors that contain all information from the original sequence.  That's exactly what we want in a fingerprint, so perhaps those embedding vectors will then be useful as a way to represent molecules in other models!

Let's start by loading the data.  We will use the MUV dataset.  It includes 74,501 molecules in the training set, and 9313 molecules in the validation set, so it gives us plenty of SMILES strings to work with.

In [17]:
# Load dataset using DeepChem
tasks, datasets, transformers = dc.molnet.load_muv(split='stratified')
train_dataset, valid_dataset, test_dataset = datasets
train_smiles = train_dataset.ids
valid_smiles = valid_dataset.ids

'split' is deprecated.  Use 'splitter' instead.


We need to define the "alphabet" for our `SeqToSeq` model, the list of all tokens that can appear in sequences.  (It's also possible for input and output sequences to have different alphabets, but since we're training it as an autoencoder, they're identical in this case.)  Make a list of every character that appears in any training sequence.

In [18]:
# Extract tokens form the SMILES strings
tokens = set()
for s in train_smiles:
  tokens = tokens.union(set(c for c in s))
tokens = sorted(list(tokens))

In [19]:
# For PyTorch, create a mapping from tokens to indices and vice versa to prepare for DataLoader
token_to_idx = {token: idx for idx, token in enumerate(tokens)}
idx_to_token = {idx: token for token, idx in token_to_idx.items()}

In [20]:
# Define PyTorch Dataset and dataloader
class SMILESDataset(Dataset):
    def __init__(self, smiles_list, token_to_idx, max_length):
        self.smiles_list = smiles_list
        self.token_to_idx = token_to_idx
        self.max_length = max_length

    def __len__(self):
        return len(self.smiles_list)

    def __getitem__(self, idx):
        smiles = self.smiles_list[idx]
        # Convert SMILES string to a sequence of token indices
        input_seq = [self.token_to_idx[token] for token in smiles]
        # Add padding to the sequence to make it `max_length`
        input_seq = input_seq + [0] * (self.max_length - len(input_seq))
        input_seq = torch.tensor(input_seq, dtype=torch.long)

        # For simplicity, use the same sequence as the target (e.g., autoencoder)
        target_seq = input_seq.clone()
        return input_seq, target_seq

# Max sequence length
max_length = max(len(s) for s in train_smiles)

# Create the training dataset and DataLoader
batch_size = 100  # Define your batch size
train_dataset = SMILESDataset(train_smiles, token_to_idx, max_length)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# Create the validation dataset and DataLoader (optional)
valid_dataset = SMILESDataset(valid_smiles, token_to_idx, max_length)
valid_dataloader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False)

Create the model and define the optimization method to use.  In this case, learning works much better if we gradually decrease the learning rate.  We use an `ExponentialDecay` to multiply the learning rate by 0.9 after each epoch.

In [21]:
# from deepchem.models.optimizers import Adam, ExponentialDecay
# import tensorflow as tf

# max_length = max(len(s) for s in train_smiles)
# batch_size = 100
# batches_per_epoch = len(train_smiles)/batch_size

# #FIXME: The Lambda layer requires an output_shape argument. Also, the SeqToSeq in dc is based on Keras.

# # Monkey patch the _create_encoder method to fix the Lambda layer
# def _create_encoder(self, n_layers, dropout):
#     from tensorflow.keras.layers import Input, Embedding, GRU, Lambda
#     input = Input(shape=(None,))
#     gather_indices = Input(shape=(2,), dtype=tf.int32)
#     prev_layer = Embedding(len(self._input_tokens),
#                           self._embedding_dimension)(input)
#     if self._reverse_input:
#         prev_layer = Lambda(lambda x: tf.reverse(x, axis=[1]))(prev_layer)
#     for i in range(n_layers):
#         if i == n_layers - 1:
#             prev_layer = GRU(self._embedding_dimension)(prev_layer)
#         else:
#             prev_layer = GRU(self._embedding_dimension,
#                              return_sequences=True)(prev_layer)
#     # Specify output_shape in the lambda function
#     prev_layer = Lambda(lambda x: tf.gather_nd(x[0], x[1]), output_shape=(self._embedding_dimension,))(
#         [prev_layer, gather_indices])
#     return tf.keras.Model(inputs=[input, gather_indices], outputs=prev_layer)


# # Monkey patch the original class
# dc.models.SeqToSeq._create_encoder = _create_encoder
# model = dc.models.SeqToSeq(tokens,
#                            tokens,
#                            max_length,
#                            encoder_layers=2,
#                            decoder_layers=2,
#                            embedding_dimension=256,
#                            model_dir='fingerprint',
#                            batch_size=batch_size,
#                            learning_rate=ExponentialDecay(0.001, 0.9, batches_per_epoch))

In [22]:
# Define the Seq2Seq model
class SeqToSeq(nn.Module):
    def __init__(self, input_vocab_size, output_vocab_size, embedding_dim, hidden_dim, max_length, num_layers):
        super(SeqToSeq, self).__init__()
        self.embedding = nn.Embedding(input_vocab_size, embedding_dim)
        self.encoder = nn.GRU(embedding_dim, hidden_dim, num_layers, batch_first=True)
        self.decoder = nn.GRU(embedding_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_vocab_size)
        self.max_length = max_length

    def forward(self, input_seq, target_seq=None):
        # Encoder
        embedded_input = self.embedding(input_seq)
        _, hidden = self.encoder(embedded_input)

        # Decoder
        if target_seq is not None:
            embedded_target = self.embedding(target_seq)
            decoder_output, _ = self.decoder(embedded_target, hidden)
        else:
            # For inference, generate sequence step by step
            decoder_input = input_seq[:, :1]  # Start with the first token
            decoder_output = []
            for _ in range(self.max_length):
                embedded_decoder_input = self.embedding(decoder_input)
                output, hidden = self.decoder(embedded_decoder_input, hidden)
                decoder_output.append(output)
                decoder_input = torch.argmax(self.fc(output), dim=-1)  # Greedy decoding
            decoder_output = torch.cat(decoder_output, dim=1)

        # Output layer
        output = self.fc(decoder_output)
        return output

# Hyperparameters
input_vocab_size = len(tokens)
output_vocab_size = len(tokens)
embedding_dim = 256
hidden_dim = 256
num_layers = 2
max_length = max(len(s) for s in train_smiles)
batch_size = 100
learning_rate = 0.001
decay_rate = 0.9
batches_per_epoch = len(train_smiles) / batch_size

# Initialize the model, optimizer, and learning rate scheduler
model = SeqToSeq(input_vocab_size, output_vocab_size, embedding_dim, hidden_dim, max_length, num_layers).to(device)
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
scheduler = ExponentialLR(optimizer, gamma=decay_rate)

# Training loop
epochs = 10
for epoch in range(epochs):
    model.train()  # Set the model to training mode
    for batch in train_dataloader:
        input_seq, target_seq = batch
        input_seq, target_seq = input_seq.to(device), target_seq.to(device)  # Move data to GPU

        optimizer.zero_grad()
        output = model(input_seq, target_seq)  # Forward pass
        loss = nn.CrossEntropyLoss()(output.view(-1, output_vocab_size), target_seq.view(-1))  # Compute loss
        loss.backward()  # Backpropagation
        optimizer.step()  # Update model parameters
    scheduler.step()  # Decay the learning rate
    print(f"Epoch {epoch + 1}, Loss: {loss.item()}")

Epoch 1, Loss: 0.0002109338529407978
Epoch 2, Loss: 6.563309580087662e-05
Epoch 3, Loss: 3.406973337405361e-05
Epoch 4, Loss: 1.4948341231502127e-05
Epoch 5, Loss: 1.277320279768901e-05
Epoch 6, Loss: 7.520345207012724e-06
Epoch 7, Loss: 6.045206191629404e-06
Epoch 8, Loss: 3.867706709570484e-06
Epoch 9, Loss: 2.7108878839499084e-06
Epoch 10, Loss: 2.088370592900901e-06


Let's train it!  The input to `fit_sequences()` is a generator that produces input/output pairs.  On a good GPU, this should take a few hours or less.

In [23]:
# def generate_sequences(epochs):
#   for i in range(epochs):
#     for s in train_smiles:
#       yield (s, s)

# model.fit_sequences(generate_sequences(40))

In [24]:
# Generate sequences from the trained model

def generate_sequence(model, input_seq, idx_to_token, max_length):
    model.eval()  # Set the model to evaluation mode
    with torch.no_grad():  # Disable gradient computation
        input_seq = torch.tensor(input_seq, dtype=torch.long).unsqueeze(0).to(device)  # Add batch dimension and move to GPU
        decoder_input = input_seq[:, :1]  # Start with the first token
        hidden = None
        generated_sequence = []

        for _ in range(max_length):
            embedded_input = model.embedding(decoder_input)
            output, hidden = model.decoder(embedded_input, hidden)
            output_token = torch.argmax(model.fc(output), dim=-1)  # Greedy decoding
            generated_sequence.append(output_token.item())
            decoder_input = output_token  # Use the output as the next input

        # Convert token indices back to SMILES string
        generated_smiles = ''.join([idx_to_token[idx] for idx in generated_sequence])
        return generated_smiles

In [25]:
# Generate input-output pairs for training (reflecting old code)
# Generate 40 sequences
generated_sequences = []
for i in range(40):
    # Select a random SMILES string from the training data
    input_smiles = train_smiles[i % len(train_smiles)]  # Loop through train_smiles if fewer than 40
    input_seq = [token_to_idx[token] for token in input_smiles]  # Convert SMILES to token indices
    input_seq = input_seq + [0] * (max_length - len(input_seq))  # Pad to max_length

    # Generate a sequence using the model
    generated_smiles = generate_sequence(model, input_seq, idx_to_token, max_length)
    generated_sequences.append(generated_smiles)

# Print the generated sequences
for i, seq in enumerate(generated_sequences):
    print(f"Generated Sequence {i + 1}: {seq}")

Generated Sequence 1: CCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCC
Generated Sequence 2: CCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCC
Generated Sequence 3: CCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCC
Generated Sequence 4: OOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOO
Generated Sequence 5: NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN
Generated Sequence 6: CCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCC
Generated Sequence 7: CCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCC
Generated Sequence 8: CCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCC
Generated Sequence 9: CCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCC
Generated Sequence 

Let's see how well it works as an autoencoder.  We'll run the first 500 molecules from the validation set through it, and see how many of them are exactly reproduced.

In [26]:
predicted = model.predict_from_sequences(valid_smiles[:500])
count = 0
for s,p in zip(valid_smiles[:500], predicted):
  if ''.join(p) == s:
    count += 1
print('reproduced', count, 'of 500 validation SMILES strings')

AttributeError: 'SeqToSeq' object has no attribute 'predict_from_sequences'

Now we'll trying using the encoder as a way to generate molecular fingerprints.  We compute the embedding vectors for all molecules in the training and validation datasets, and create new datasets that have those as their feature vectors.  The amount of data is small enough that we can just store everything in memory.

In [ ]:
import numpy as np
train_embeddings = model.predict_embeddings(train_smiles)
train_embeddings_dataset = dc.data.NumpyDataset(train_embeddings,
                                                train_dataset.y,
                                                train_dataset.w.astype(np.float32),
                                                train_dataset.ids)

valid_embeddings = model.predict_embeddings(valid_smiles)
valid_embeddings_dataset = dc.data.NumpyDataset(valid_embeddings,
                                                valid_dataset.y,
                                                valid_dataset.w.astype(np.float32),
                                                valid_dataset.ids)

For classification, we'll use a simple fully connected network with one hidden layer.

In [ ]:
classifier = dc.models.MultitaskClassifier(n_tasks=len(tasks),
                                                      n_features=256,
                                                      layer_sizes=[512])
classifier.fit(train_embeddings_dataset, nb_epoch=10)

Find out how well it worked.  Compute the ROC AUC for the training and validation datasets.

In [ ]:
metric = dc.metrics.Metric(dc.metrics.roc_auc_score, np.mean, mode="classification")
train_score = classifier.evaluate(train_embeddings_dataset, [metric], transformers)
valid_score = classifier.evaluate(valid_embeddings_dataset, [metric], transformers)
print('Training set ROC AUC:', train_score)
print('Validation set ROC AUC:', valid_score)

# Congratulations! Time to join the Community!

Congratulations on completing this tutorial notebook! If you enjoyed working through the tutorial, and want to continue working with DeepChem, we encourage you to finish the rest of the tutorials in this series. You can also help the DeepChem community in the following ways:

## Star DeepChem on [GitHub](https://github.com/deepchem/deepchem)
This helps build awareness of the DeepChem project and the tools for open source drug discovery that we're trying to build.

## Join the DeepChem Gitter
The DeepChem [Gitter](https://gitter.im/deepchem/Lobby) hosts a number of scientists, developers, and enthusiasts interested in deep learning for the life sciences. Join the conversation!